# Mortgage Refinancing Exposure Analytics - SQL Analysis

This notebook uses the validated loan-month dataset produced by the Python analytical pipeline as the SQL analytical layer.

The curated dataset contains **9,647,257 loan-month observations across 295,750 mortgages**, covering January 2019 through March 2026.

The SQL analysis focuses on historical prepayment behaviour, current portfolio exposure, interest-rate scenario analysis, and exposure concentration.

## Analytical Data Handoff

The validated loan-month dataset produced by the Python analytical pipeline is persisted as a curated Parquet dataset and registered as a SQL table for downstream analysis.

The SQL layer independently validates the analytical population before reproducing and extending the core portfolio analysis.

In [0]:
%sql
CREATE OR REPLACE TABLE mortgage_refinancing.mortgage_analysis.loan_month
AS

SELECT *
FROM parquet.`/Volumes/mortgage_refinancing/mortgage_analysis/curated_data/loan_month.parquet`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_loan_months,
    COUNT(DISTINCT loan_identifier) AS unique_loans,
    MIN(period) AS first_period,
    MAX(period) AS last_period
FROM mortgage_refinancing.mortgage_analysis.loan_month;

total_loan_months,unique_loans,first_period,last_period
9647257,295750,2019-01-01T00:00:00.000,2026-03-01T00:00:00.000


# 1. PORTFOLIO DATA FOUNDATION

The curated loan-month dataset produced by the Python analytical pipeline is used as the SQL analytical layer.

This section validates the analytical population and establishes the portfolio-level structure used in subsequent refinancing exposure analysis.

## 1.1. Analytical Population by Vintage

Summarize the analytical population by origination vintage to verify loan coverage and the volume of monthly performance observations available for subsequent analysis.

In [0]:
%sql
SELECT
    vintage,
    COUNT(DISTINCT loan_identifier) AS unique_loans,
    COUNT(*) AS loan_months
FROM mortgage_refinancing.mortgage_analysis.loan_month
GROUP BY vintage
ORDER BY vintage;

vintage,unique_loans,loan_months
2019,41250,1562943
2020,36536,1831380
2021,37041,1873772
2022,43697,1769943
2023,46686,1362335
2024,46362,885815
2025,44178,361069


## 1.2. Validate Loan-Month Grain

Verify that each mortgage has at most one performance observation per reporting month, confirming that the dataset operates at the intended loan-month grain.

In [0]:
%sql
SELECT
    loan_identifier,
    period,
    COUNT(*) AS row_count
FROM mortgage_refinancing.mortgage_analysis.loan_month
GROUP BY
    loan_identifier,
    period
HAVING COUNT(*) > 1;

loan_identifier,period,row_count


## 1.3. Validate Voluntary Payoff Events

Validate the voluntary-payoff indicator by confirming the total number of observed events and ensuring that no mortgage records more than one voluntary payoff event.

In [0]:
%sql
WITH payoff_by_loan AS (
    SELECT
        loan_identifier,
        SUM(voluntary_payoff_flag) AS payoff_events
    FROM mortgage_refinancing.mortgage_analysis.loan_month
    GROUP BY loan_identifier
)

SELECT
    SUM(payoff_events) AS voluntary_payoff_events,
    SUM(CASE WHEN payoff_events > 0 THEN 1 ELSE 0 END) AS loans_with_voluntary_payoff,
    MAX(payoff_events) AS max_payoff_events_per_loan
FROM payoff_by_loan;

voluntary_payoff_events,loans_with_voluntary_payoff,max_payoff_events_per_loan
83225,83225,1


# 2. HISTORICAL PREPAYMENT BEHAVIOUR 

Historical loan-month observations are used to evaluate how voluntary payoff behaviour changes as the economic incentive to refinance increases.

The analysis compares voluntary payoff rates across refinancing-incentive segments to establish an evidence-based monitoring threshold for subsequent portfolio scenario analysis.

## 2.1. Voluntary Payoff Rate by Refinancing Incentive

Compare historical voluntary payoff behaviour between loan-months with positive and non-positive refinancing incentives.

In [0]:
%sql
SELECT
    CASE
        WHEN rate_incentive_bps > 0 THEN 'Positive incentive'
        ELSE 'Non-positive incentive'
    END AS incentive_group,

    COUNT(*) AS loan_months,

    SUM(voluntary_payoff_flag) AS voluntary_payoffs,

    ROUND(
        100.0 * SUM(voluntary_payoff_flag) / COUNT(*),
        3
    ) AS payoff_rate_pct

FROM mortgage_refinancing.mortgage_analysis.loan_month

GROUP BY
    CASE
        WHEN rate_incentive_bps > 0 THEN 'Positive incentive'
        ELSE 'Non-positive incentive'
    END

ORDER BY payoff_rate_pct DESC;

incentive_group,loan_months,voluntary_payoffs,payoff_rate_pct
Positive incentive,3196041,54987,1.720
Non-positive incentive,6451216,28238,0.438


## 2.2. Voluntary Payoff Rate Around the 50-bps Monitoring Threshold

Compare historical voluntary payoff behaviour below and above the 50-basis-point refinancing-incentive threshold used for portfolio monitoring.

In [0]:
%sql
SELECT
    CASE
        WHEN rate_incentive_bps >= 50 THEN '50+ bps'
        ELSE '<50 bps'
    END AS incentive_group,

    COUNT(*) AS loan_months,

    SUM(voluntary_payoff_flag) AS voluntary_payoffs,

    ROUND(
        100.0 * SUM(voluntary_payoff_flag) / COUNT(*),
        3
    ) AS payoff_rate_pct

FROM mortgage_refinancing.mortgage_analysis.loan_month

GROUP BY
    CASE
        WHEN rate_incentive_bps >= 50 THEN '50+ bps'
        ELSE '<50 bps'
    END

ORDER BY payoff_rate_pct DESC;

incentive_group,loan_months,voluntary_payoffs,payoff_rate_pct
50+ bps,1869195,44076,2.358
<50 bps,7778062,39149,0.503


## 2.3. Voluntary Payoff Rate Across Refinancing-Incentive Bands

Segment historical loan-month observations into refinancing-incentive bands to examine how voluntary payoff behaviour changes as the economic incentive to refinance increases.

In [0]:
%sql
WITH incentive_bands AS (
    SELECT
        CASE
            WHEN rate_incentive_bps < 0 THEN '<0 bps'
            WHEN rate_incentive_bps < 50 THEN '0-49 bps'
            WHEN rate_incentive_bps < 100 THEN '50-99 bps'
            WHEN rate_incentive_bps < 150 THEN '100-149 bps'
            ELSE '150+ bps'
        END AS incentive_band,

        CASE
            WHEN rate_incentive_bps < 0 THEN 1
            WHEN rate_incentive_bps < 50 THEN 2
            WHEN rate_incentive_bps < 100 THEN 3
            WHEN rate_incentive_bps < 150 THEN 4
            ELSE 5
        END AS band_order,

        voluntary_payoff_flag

    FROM mortgage_refinancing.mortgage_analysis.loan_month
)

SELECT
    incentive_band,
    COUNT(*) AS loan_months,
    SUM(voluntary_payoff_flag) AS voluntary_payoffs,
    ROUND(
        100.0 * SUM(voluntary_payoff_flag) / COUNT(*),
        3
    ) AS payoff_rate_pct

FROM incentive_bands

GROUP BY
    incentive_band,
    band_order

ORDER BY band_order;

incentive_band,loan_months,voluntary_payoffs,payoff_rate_pct
<0 bps,6444421,28212,0.438
0-49 bps,1333641,10937,0.820
50-99 bps,1057946,19005,1.796
100-149 bps,514776,15082,2.930
150+ bps,296473,9989,3.369


- Observed voluntary payoff rates increase progressively across refinancing-incentive bands, from 0.438% for loan-months below 0 bps to 3.369% for observations at 150+ bps.

- The observed increase beyond 50 bps supports its use as a practical portfolio-monitoring threshold in the subsequent scenario analysis. The threshold is analytical rather than an industry-standard or predictive cutoff, and the historical relationship should not be interpreted as causal.

# 3. CURRENT PORTFOLIO SNAPSHOT

The latest available reporting period, March 2026, is used to construct the active portfolio snapshot.

Only mortgages with positive current outstanding principal are retained. Refinancing sensitivity is defined using the 50-basis-point monitoring threshold established from the historical analysis.

## 3.1. March 2026 Active Portfolio

Measure the size and outstanding principal balance of the active mortgage portfolio at the latest available reporting period.

In [0]:
%sql
WITH active_portfolio AS (
    SELECT *
    FROM mortgage_refinancing.mortgage_analysis.loan_month
    WHERE period = DATE '2026-03-01'
      AND current_actual_upb > 0
)

SELECT
    COUNT(*) AS active_loans,
    ROUND(SUM(current_actual_upb) / 1000000000.0, 2) AS active_upb_bn
FROM active_portfolio;

active_loans,active_upb_bn
211265,62.14


## 3.2. Current Rate-Sensitive Exposure

Quantify the portion of the March 2026 active portfolio currently exceeding the 50-basis-point refinancing-incentive monitoring threshold.

In [0]:
%sql
WITH active_portfolio AS (
    SELECT *
    FROM mortgage_refinancing.mortgage_analysis.loan_month
    WHERE period = DATE '2026-03-01'
      AND current_actual_upb > 0
)

SELECT
    COUNT(*) AS active_loans,

    SUM(
        CASE WHEN rate_incentive_bps >= 50 THEN 1 ELSE 0 END
    ) AS rate_sensitive_loans,

    ROUND(
        SUM(
            CASE WHEN rate_incentive_bps >= 50
                 THEN current_actual_upb ELSE 0 END
        ) / 1000000000.0,
        2
    ) AS rate_sensitive_upb_bn,

    ROUND(
        100.0 * SUM(
            CASE WHEN rate_incentive_bps >= 50
                 THEN current_actual_upb ELSE 0 END
        ) / SUM(current_actual_upb),
        2
    ) AS exposure_pct

FROM active_portfolio;

active_loans,rate_sensitive_loans,rate_sensitive_upb_bn,exposure_pct
211265,54265,16.31,26.25


# 4. INTEREST-RATE SCENARIO ANALYSIS 

Stress-test the March 2026 active mortgage portfolio under hypothetical declines in the 30-year market mortgage rate.

For each scenario, a mortgage is classified as rate-sensitive when its current mortgage rate exceeds the shocked market rate by at least 50 basis points. The resulting balances represent portfolio exposure, not predicted prepayments or losses.

## 4.1. Portfolio Exposure Under Falling-Rate Scenarios

Evaluate how much active mortgage UPB crosses the 50-basis-point monitoring threshold under the current market rate and hypothetical declines of 50, 100, and 150 basis points.

In [0]:
%sql
WITH active_portfolio AS (
    SELECT *
    FROM mortgage_refinancing.mortgage_analysis.loan_month
    WHERE period = DATE '2026-03-01'
      AND current_actual_upb > 0
),

scenarios AS (
    SELECT 'Current' AS scenario, 6.178 AS market_rate, 1 AS scenario_order
    UNION ALL
    SELECT '-50 bps', 5.678, 2
    UNION ALL
    SELECT '-100 bps', 5.178, 3
    UNION ALL
    SELECT '-150 bps', 4.678, 4
)

SELECT
    s.scenario,
    s.market_rate,

    SUM(
        CASE WHEN (a.current_interest_rate - s.market_rate) * 100 >= 50
             THEN 1 ELSE 0 END
    ) AS rate_sensitive_loans,

    ROUND(
        SUM(
            CASE WHEN (a.current_interest_rate - s.market_rate) * 100 >= 50
                 THEN a.current_actual_upb ELSE 0 END
        ) / 1000000000.0,
        2
    ) AS rate_sensitive_upb_bn,

    ROUND(
        100.0 * SUM(
            CASE WHEN (a.current_interest_rate - s.market_rate) * 100 >= 50
                 THEN a.current_actual_upb ELSE 0 END
        ) / SUM(a.current_actual_upb),
        2
    ) AS exposure_pct

FROM active_portfolio a
CROSS JOIN scenarios s

GROUP BY
    s.scenario,
    s.market_rate,
    s.scenario_order

ORDER BY s.scenario_order;

scenario,market_rate,rate_sensitive_loans,rate_sensitive_upb_bn,exposure_pct
Current,6.178,54265,16.31,26.25
-50 bps,5.678,94571,29.2,46.98
-100 bps,5.178,120220,37.56,60.44
-150 bps,4.678,131024,40.84,65.71


## 4.2. Incremental Exposure Across Rate-Shock Scenarios

Measure the additional mortgage UPB entering the rate-sensitive population as market rates decline across successive scenarios.

In [0]:
%sql
WITH active_portfolio AS (
    SELECT *
    FROM mortgage_refinancing.mortgage_analysis.loan_month
    WHERE period = DATE '2026-03-01'
      AND current_actual_upb > 0
),

scenarios AS (
    SELECT 'Current' AS scenario, 6.178 AS market_rate, 1 AS scenario_order
    UNION ALL
    SELECT '-50 bps', 5.678, 2
    UNION ALL
    SELECT '-100 bps', 5.178, 3
    UNION ALL
    SELECT '-150 bps', 4.678, 4
),

scenario_exposure AS (
    SELECT
        s.scenario,
        s.scenario_order,

        SUM(
            CASE WHEN (a.current_interest_rate - s.market_rate) * 100 >= 50
                 THEN a.current_actual_upb ELSE 0 END
        ) / 1000000000.0 AS sensitive_upb_bn

    FROM active_portfolio a
    CROSS JOIN scenarios s

    GROUP BY
        s.scenario,
        s.scenario_order
)

SELECT
    scenario,
    ROUND(sensitive_upb_bn, 2) AS sensitive_upb_bn,

    ROUND(
        sensitive_upb_bn
        - LAG(sensitive_upb_bn) OVER (ORDER BY scenario_order),
        2
    ) AS incremental_upb_bn

FROM scenario_exposure

ORDER BY scenario_order;

scenario,sensitive_upb_bn,incremental_upb_bn
Current,16.31,null
-50 bps,29.2,12.89
-100 bps,37.56,8.37
-150 bps,40.84,3.28


- Rate-sensitive exposure increases materially as market rates decline, but the incremental exposure added by each successive 50-bps decline becomes smaller.

- The first 50-bps decline adds approximately $12.89B of rate-sensitive UPB, followed by $8.37B under the next 50-bps decline and $3.28B under the final step.

- This indicates that a substantial portion of the portfolio becomes rate-sensitive relatively early in the falling-rate scenarios, while deeper rate declines bring progressively smaller additional balances above the monitoring threshold.

# 5. EXPOSURE CONCENTRATION

The -100-bps scenario is used as a representative falling-rate stress scenario to examine where rate-sensitive exposure is concentrated within the active portfolio.

Exposure is evaluated across borrower and loan characteristics to distinguish between the dollar concentration of sensitive balances and the rate-sensitivity intensity within each segment.

## 5.1. Rate-Sensitive Exposure by Current LTV

Segment the March 2026 active portfolio by estimated current loan-to-value ratio (LTV) and measure both the dollar amount and intensity of rate-sensitive exposure under the -100-bps scenario.

In [0]:
%sql
WITH active_portfolio AS (
    SELECT
        *,
        CASE
            WHEN estimated_ltv < 60 THEN '<60'
            WHEN estimated_ltv < 70 THEN '60-69'
            WHEN estimated_ltv < 80 THEN '70-79'
            WHEN estimated_ltv < 90 THEN '80-89'
            WHEN estimated_ltv < 100 THEN '90-99'
            ELSE '100+'
        END AS ltv_band,

        CASE
            WHEN estimated_ltv < 60 THEN 1
            WHEN estimated_ltv < 70 THEN 2
            WHEN estimated_ltv < 80 THEN 3
            WHEN estimated_ltv < 90 THEN 4
            WHEN estimated_ltv < 100 THEN 5
            ELSE 6
        END AS ltv_order

    FROM mortgage_refinancing.mortgage_analysis.loan_month
    WHERE period = DATE '2026-03-01'
      AND current_actual_upb > 0
      AND estimated_ltv IS NOT NULL
),

ltv_exposure AS (
    SELECT
        ltv_band,
        ltv_order,
        COUNT(*) AS total_loans,
        SUM(current_actual_upb) AS total_upb,

        SUM(
            CASE WHEN (current_interest_rate - 5.178) * 100 >= 50
                 THEN 1 ELSE 0 END
        ) AS rate_sensitive_loans,

        SUM(
            CASE WHEN (current_interest_rate - 5.178) * 100 >= 50
                 THEN current_actual_upb ELSE 0 END
        ) AS rate_sensitive_upb

    FROM active_portfolio

    GROUP BY
        ltv_band,
        ltv_order
)

SELECT
    ltv_band,
    total_loans,
    ROUND(total_upb / 1000000000.0, 2) AS total_upb_bn,
    rate_sensitive_loans,
    ROUND(rate_sensitive_upb / 1000000000.0, 2) AS rate_sensitive_upb_bn,

    ROUND(
        100.0 * rate_sensitive_upb
        / SUM(rate_sensitive_upb) OVER (),
        2
    ) AS share_of_sensitive_upb_pct,

    ROUND(
        100.0 * rate_sensitive_upb / total_upb,
        2
    ) AS exposure_intensity_pct

FROM ltv_exposure

ORDER BY ltv_order;

ltv_band,total_loans,total_upb_bn,rate_sensitive_loans,rate_sensitive_upb_bn,share_of_sensitive_upb_pct,exposure_intensity_pct
<60,84058,20.14,26605,6.11,17.27,30.33
60-69,30636,9.3,16592,5.06,14.3,54.39
70-79,36102,12.19,27624,9.33,26.38,76.54
80-89,28069,10.06,23913,8.52,24.11,84.71
90-99,16772,6.35,15243,5.77,16.31,90.75
100+,1899,0.68,1633,0.58,1.63,85.22


- Under the -100-bps scenario, the 70–79 LTV segment contains the largest concentration of rate-sensitive UPB at approximately $9.33B, representing 26.38% of total sensitive exposure.

- Exposure intensity rises across the LTV bands through 90–99, reaching 90.81%, before declining to 85.02% for loans at 100+ LTV.

- These results describe portfolio concentration rather than a causal relationship between LTV and refinancing sensitivity. Differences across LTV segments may also reflect underlying vintage and mortgage-rate composition.

## 5.2. Rate-Sensitive Exposure by FICO

Evaluate rate-sensitive exposure across borrower credit-score segments under the -100-bps scenario to assess whether FICO provides meaningful additional differentiation in portfolio rate sensitivity.

In [0]:
%sql
WITH active_portfolio AS (
    SELECT
        *,
        CASE
            WHEN classic_fico < 660 THEN '<660'
            WHEN classic_fico < 700 THEN '660-699'
            WHEN classic_fico < 740 THEN '700-739'
            WHEN classic_fico < 780 THEN '740-779'
            ELSE '780+'
        END AS fico_band,

        CASE
            WHEN classic_fico < 660 THEN 1
            WHEN classic_fico < 700 THEN 2
            WHEN classic_fico < 740 THEN 3
            WHEN classic_fico < 780 THEN 4
            ELSE 5
        END AS fico_order

    FROM mortgage_refinancing.mortgage_analysis.loan_month
    WHERE period = DATE '2026-03-01'
      AND current_actual_upb > 0
      AND classic_fico IS NOT NULL
),

fico_exposure AS (
    SELECT
        fico_band,
        fico_order,
        COUNT(*) AS total_loans,
        SUM(current_actual_upb) AS total_upb,

        SUM(
            CASE WHEN (current_interest_rate - 5.178) * 100 >= 50
                 THEN 1 ELSE 0 END
        ) AS rate_sensitive_loans,

        SUM(
            CASE WHEN (current_interest_rate - 5.178) * 100 >= 50
                 THEN current_actual_upb ELSE 0 END
        ) AS rate_sensitive_upb

    FROM active_portfolio

    GROUP BY
        fico_band,
        fico_order
)

SELECT
    fico_band,
    total_loans,
    ROUND(total_upb / 1000000000.0, 2) AS total_upb_bn,
    rate_sensitive_loans,
    ROUND(rate_sensitive_upb / 1000000000.0, 2) AS rate_sensitive_upb_bn,

    ROUND(
        100.0 * rate_sensitive_upb
        / SUM(rate_sensitive_upb) OVER (),
        2
    ) AS share_of_sensitive_upb_pct,

    ROUND(
        100.0 * rate_sensitive_upb / total_upb,
        2
    ) AS exposure_intensity_pct

FROM fico_exposure

ORDER BY fico_order;

fico_band,total_loans,total_upb_bn,rate_sensitive_loans,rate_sensitive_upb_bn,share_of_sensitive_upb_pct,exposure_intensity_pct
<660,7544,1.78,4723,1.17,3.12,65.76
660-699,22543,5.89,12917,3.56,9.47,60.37
700-739,44732,12.76,25156,7.59,20.22,59.52
740-779,68495,21.06,38938,12.73,33.89,60.42
780+,67900,20.64,38449,12.5,33.3,60.59


- Under the -100-bps scenario, rate-sensitive exposure is concentrated in the higher-FICO segments in dollar terms, with the 740–779 and 780+ bands together accounting for approximately two-thirds of sensitive UPB.

- However, exposure intensity is relatively similar across FICO bands at 660 and above, remaining around 60%. The <660 segment shows somewhat higher intensity at 65.76% but represents only 3.12% of total sensitive UPB.

- This suggests that FICO provides limited differentiation in refinancing rate sensitivity within this portfolio compared with the more pronounced segmentation observed across current-LTV bands.

# 6. FINAL ANALYTICAL SUMMARY

The SQL analysis independently reproduces the core portfolio findings established in the Python analytical workflow and extends them through structured segmentation and scenario analysis.

## 6.1. Key Findings

- Historical voluntary payoff rates increase as refinancing incentive rises, from **0.438% below 0 bps** to **3.369% at 150+ bps**.
- Loan-months with at least **50 bps of refinancing incentive** recorded a **2.358% voluntary payoff rate**, compared with **0.503% below the threshold**, supporting 50 bps as a practical monitoring threshold.
- As of **March 2026**, the active portfolio contains **211,265 loans** and approximately **$62.14B of outstanding UPB**.
- At the current 30-year market rate of **6.178%**, approximately **$16.31B**, or **26.25%**, of active UPB is rate-sensitive.
- Under a **100-bps decline** in market rates, rate-sensitive exposure increases to approximately **$37.56B**, representing **60.44%** of active portfolio UPB.
- Incremental exposure is largest in the first 50-bps decline (**+$12.89B**) and becomes progressively smaller under deeper rate declines.
- Under the -100-bps scenario, the **70–79 LTV** segment contains the largest dollar concentration of rate-sensitive exposure (**$9.33B**), while the **90–99 LTV** segment has the highest exposure intensity (**90.81%**).
- FICO provides comparatively limited differentiation in rate sensitivity among borrowers with scores of 660 or above, where exposure intensity remains approximately **60%**.

## 6.2. Interpretation

- The analysis indicates that falling mortgage rates can materially expand the portion of the portfolio exposed to refinancing incentives. The exposure is not distributed uniformly across loan characteristics, making portfolio segmentation useful for identifying where rate-sensitive balances are concentrated.

- These results represent **portfolio exposure under hypothetical rate scenarios**, not predicted borrower-level refinancing behaviour, expected prepayments, or financial losses.